# Data Understanding

## Objective

Before any cleaning or modelling, confirm the structure of the raw Walmart sales data: what each file contains, how the files relate to each other, and whether the data matches what the source documentation claims. This determines what cleaning is actually needed in the next notebook.


In [1]:
import pandas as pd

train = pd.read_csv("../data/raw/train.csv")
features = pd.read_csv("../data/raw/features.csv")
stores = pd.read_csv("../data/raw/stores.csv")

print("train shape:", train.shape)
print("features shape:", features.shape)
print("stores shape:", stores.shape)

train shape: (421570, 5)
features shape: (8190, 12)
stores shape: (45, 3)


In [3]:
print("TRAIN")
print(train.dtypes)
print()
print("FEATURES")
print(features.dtypes)
print()
print("STORES")
print(stores.dtypes)

TRAIN
Store             int64
Dept              int64
Date             object
Weekly_Sales    float64
IsHoliday          bool
dtype: object

FEATURES
Store             int64
Date             object
Temperature     float64
Fuel_Price      float64
MarkDown1       float64
MarkDown2       float64
MarkDown3       float64
MarkDown4       float64
MarkDown5       float64
CPI             float64
Unemployment    float64
IsHoliday          bool
dtype: object

STORES
Store     int64
Type     object
Size      int64
dtype: object


In [5]:
train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


In [7]:
features.head()

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [9]:
stores.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [11]:
train.shape

(421570, 5)

## Data Validation

### Missing Values


In [13]:
print("TRAIN missing values")
print(train.isnull().sum())
print()
print("FEATURES missing values")
print(features.isnull().sum())
print()
print("STORES missing values")
print(stores.isnull().sum())

TRAIN missing values
Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
dtype: int64

FEATURES missing values
Store              0
Date               0
Temperature        0
Fuel_Price         0
MarkDown1       4158
MarkDown2       5269
MarkDown3       4577
MarkDown4       4726
MarkDown5       4140
CPI              585
Unemployment     585
IsHoliday          0
dtype: int64

STORES missing values
Store    0
Type     0
Size     0
dtype: int64


In [15]:
missing_pct = features[["MarkDown1","MarkDown2","MarkDown3","MarkDown4","MarkDown5"]].isnull().mean() * 100
print(missing_pct.round(1))

MarkDown1    50.8
MarkDown2    64.3
MarkDown3    55.9
MarkDown4    57.7
MarkDown5    50.5
dtype: float64


### Duplicate Records

In [17]:
print("train duplicate rows:", train.duplicated().sum())
print("train duplicate Store-Dept-Date combos:", train.duplicated(subset=["Store","Dept","Date"]).sum())
print()
print("features duplicate rows:", features.duplicated().sum())
print("features duplicate Store-Date combos:", features.duplicated(subset=["Store","Date"]).sum())
print()
print("stores duplicate rows:", stores.duplicated().sum())
print("stores duplicate Store IDs:", stores.duplicated(subset=["Store"]).sum())

train duplicate rows: 0
train duplicate Store-Dept-Date combos: 0

features duplicate rows: 0
features duplicate Store-Date combos: 0

stores duplicate rows: 0
stores duplicate Store IDs: 0


### Date Consistency

In [19]:
train["Date"] = pd.to_datetime(train["Date"])
features["Date"] = pd.to_datetime(features["Date"])

print("train date range:", train["Date"].min(), "to", train["Date"].max())
print("features date range:", features["Date"].min(), "to", features["Date"].max())

expected_weeks = pd.date_range(start=train["Date"].min(), end=train["Date"].max(), freq="W-FRI")
actual_weeks = sorted(train["Date"].unique())

print("expected number of weeks:", len(expected_weeks))
print("actual number of unique weeks in train:", len(actual_weeks))

missing_weeks = set(expected_weeks) - set(actual_weeks)
print("missing weeks:", sorted(missing_weeks))

train date range: 2010-02-05 00:00:00 to 2012-10-26 00:00:00
features date range: 2010-02-05 00:00:00 to 2013-07-26 00:00:00
expected number of weeks: 143
actual number of unique weeks in train: 143
missing weeks: []


In [21]:
print("Weekly_Sales min:", train["Weekly_Sales"].min())
print("Weekly_Sales max:", train["Weekly_Sales"].max())
print("Number of negative Weekly_Sales rows:", (train["Weekly_Sales"] < 0).sum())
print("Number of zero Weekly_Sales rows:", (train["Weekly_Sales"] == 0).sum())

Weekly_Sales min: -4988.94
Weekly_Sales max: 693099.36
Number of negative Weekly_Sales rows: 1285
Number of zero Weekly_Sales rows: 73


## Findings

**Structure:** train (421,570 rows: Store, Dept, Date, Weekly_Sales, IsHoliday), features (8,190 rows: Store, Date, Temperature, Fuel_Price, MarkDown1-5, CPI, Unemployment, IsHoliday), stores (45 rows: Store, Type, Size).

**Date coverage:** train runs 2010-02-05 to 2012-10-26 (143 weeks, no missing weeks at the calendar level). features runs 2010-02-05 to 2013-07-26 — it extends beyond train's range, which will supply external features for the future forecast period.

**Duplicates:** none in any file (train has no duplicate Store-Dept-Date combos, features no duplicate Store-Date combos, stores no duplicate Store IDs).

**Missing values:** train and stores have none. features has substantial missingness in MarkDown1-5 (50.8% to 64.3% missing) and a smaller gap in CPI/Unemployment (585 rows, ~7% of features rows). The MarkDown missingness is treated as a real absence of tracked promotional activity for much of the period, not a data error — this will need a defensible imputation/flag strategy before featurization, not a blind fillna(0).

**Weekly_Sales sanity:** ranges from -4,988.94 to 693,099.36. 1,285 rows (0.3%) are negative and 73 rows are exactly zero. Negative values plausibly represent returns exceeding sales for a store-department-week; these will not be deleted outright without further per-store/per-dept investigation in the cleaning notebook.

**Implication for cleaning notebook:** MarkDown missingness needs an explicit strategy (likely: treat missing as "no active markdown" rather than imputing a numeric value). Negative Weekly_Sales values need a closer look before deciding to keep, cap, or flag them.